# TCP/IP Encapsulation Project

**Students:** Adir Buksila & Liav Wizman

**Goal:** Craft IPv4+TCP packets, send them over loopback, and capture in Wireshark.

**Flow:** CSV (Application Messages) → Notebook (Encapsulation Simulation) → Wireshark (Capture) → Report

---

## Prerequisites (Windows)

1. Install **Wireshark** with **Npcap** (enable *WinPcap API-compatible mode* + *Support loopback traffic*)
2. Run: `pip install scapy pandas`
3. Run Jupyter **as Administrator**
4. In Wireshark, capture on **Npcap Loopback Adapter**


In [ ]:
# Step 1: Import Libraries and Check Environment
import socket
import struct
import random
import time
import platform
import pandas as pd
from typing import Optional

# Check if running on Windows
IS_WINDOWS = (platform.system() == 'Windows')
print(f"Operating System: {platform.system()}")
print(f"Running on Windows: {IS_WINDOWS}")

# Try to import Scapy (required for Windows)
try:
    from scapy.all import IP as SCAPY_IP, TCP as SCAPY_TCP, Raw as SCAPY_Raw, send as scapy_send, get_if_list
    HAVE_SCAPY = True
    print("✓ Scapy is available")
except Exception as e:
    HAVE_SCAPY = False
    SCAPY_IMPORT_ERR = e
    print(f"✗ Scapy not available: {e}")

print(f"\nEnvironment Ready: {IS_WINDOWS and HAVE_SCAPY or not IS_WINDOWS}")


In [ ]:
# Step 2: Load CSV File with Messages
CSV_PATH = "./adir_liav_http_input.csv"
messages_df = pd.read_csv(CSV_PATH)

# Display the loaded data
print(f"Loaded {len(messages_df)} messages from {CSV_PATH}")
print("\n=== CSV Preview ===")
messages_df


In [ ]:
# Step 3: Validate CSV Schema
def validate_csv_format(df: pd.DataFrame):
    """Validate that the CSV has all required columns."""
    expected_columns = ["app_protocol", "src_port", "dst_port", "message", "timestamp"]
    missing = []
    for col in expected_columns:
        if col not in df.columns:
            missing.append(col)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    print("✓ CSV schema is valid!")
    print(f"  Columns found: {list(df.columns)}")

# Fill NaN messages with empty strings
messages_df['message'] = messages_df['message'].fillna('')

# Validate
validate_csv_format(messages_df)


In [ ]:
# Step 4: Helper Functions (Checksum & Hexdump)

def checksum(data: bytes) -> int:
    """
    Calculate the Internet checksum (RFC 1071).
    Used for IP and TCP header checksums.
    """
    if len(data) % 2:
        data += b'\0'  # Pad to even length
    res = sum(struct.unpack('!%dH' % (len(data) // 2), data))
    while res >> 16:
        res = (res & 0xFFFF) + (res >> 16)
    return ~res & 0xFFFF


def hexdump(data: bytes, width: int = 16):
    """
    Display bytes in a hex dump format (like Wireshark).
    Shows offset, hex bytes, and ASCII representation.
    """
    for i in range(0, len(data), width):
        chunk = data[i:i + width]
        hex_bytes = ' '.join(f'{b:02x}' for b in chunk)
        ascii_bytes = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f"{i:04x}  {hex_bytes:<{width * 3}}  {ascii_bytes}")

print("✓ Helper functions defined")


In [ ]:
# Step 5: Build IP Header (Internet Layer)

def build_ip_header(src_ip: str, dst_ip: str, payload_len: int, proto: int = socket.IPPROTO_TCP) -> bytes:
    """
    Build an IPv4 header.
    
    IPv4 Header Structure (20 bytes):
    - Version (4 bits) + IHL (4 bits)
    - Type of Service (8 bits)
    - Total Length (16 bits)
    - Identification (16 bits)
    - Flags (3 bits) + Fragment Offset (13 bits)
    - TTL (8 bits)
    - Protocol (8 bits)
    - Header Checksum (16 bits)
    - Source IP (32 bits)
    - Destination IP (32 bits)
    """
    version_ihl = (4 << 4) + 5  # IPv4, 5 32-bit words (20 bytes)
    tos = 0                     # Type of Service
    total_length = 20 + payload_len
    identification = random.randint(0, 65535)
    flags_fragment = 0          # No fragmentation
    ttl = 64                    # Time to Live
    header_checksum = 0         # Will be calculated
    src = socket.inet_aton(src_ip)
    dst = socket.inet_aton(dst_ip)
    
    # Build header with zero checksum first
    ip_header = struct.pack('!BBHHHBBH4s4s',
                            version_ihl, tos, total_length, identification,
                            flags_fragment, ttl, proto, header_checksum,
                            src, dst)
    
    # Calculate and insert checksum
    chksum = checksum(ip_header)
    ip_header = struct.pack('!BBHHHBBH4s4s',
                            version_ihl, tos, total_length, identification,
                            flags_fragment, ttl, proto, chksum,
                            src, dst)
    return ip_header

print("✓ IP header builder defined")


In [ ]:
# Step 6: Build TCP Header (Transport Layer)

def build_tcp_header(src_ip: str, dst_ip: str, src_port: int, dst_port: int, 
                     payload: bytes = b'', seq: Optional[int] = None, 
                     ack_seq: int = 0, flags: int = 0x02, window: int = 65535) -> bytes:
    """
    Build a TCP header with proper checksum.
    
    TCP Header Structure (20 bytes minimum):
    - Source Port (16 bits)
    - Destination Port (16 bits)
    - Sequence Number (32 bits)
    - Acknowledgment Number (32 bits)
    - Data Offset (4 bits) + Reserved (4 bits)
    - Flags (8 bits): CWR, ECE, URG, ACK, PSH, RST, SYN, FIN
    - Window Size (16 bits)
    - Checksum (16 bits)
    - Urgent Pointer (16 bits)
    
    Common Flag Values:
    - 0x02 = SYN
    - 0x10 = ACK
    - 0x18 = PSH+ACK
    - 0x12 = SYN+ACK
    - 0x01 = FIN
    - 0x04 = RST
    """
    if seq is None:
        seq = random.randint(0, 0xFFFFFFFF)
    
    doff_reserved = (5 << 4)  # Data offset: 5 32-bit words (20 bytes)
    checksum_tcp = 0
    urg_ptr = 0
    
    # Build TCP header with zero checksum
    tcp_header = struct.pack('!HHLLBBHHH',
                             src_port, dst_port, seq, ack_seq,
                             doff_reserved, flags, window,
                             checksum_tcp, urg_ptr)
    
    # Build pseudo-header for checksum calculation
    placeholder = 0
    protocol = socket.IPPROTO_TCP
    tcp_length = len(tcp_header) + len(payload)
    pseudo_header = struct.pack('!4s4sBBH',
                                socket.inet_aton(src_ip), socket.inet_aton(dst_ip),
                                placeholder, protocol, tcp_length)
    
    # Calculate checksum over pseudo-header + TCP header + payload
    chksum = checksum(pseudo_header + tcp_header + payload)
    
    # Rebuild with correct checksum
    tcp_header = struct.pack('!HHLLBBHHH',
                             src_port, dst_port, seq, ack_seq,
                             doff_reserved, flags, window,
                             chksum, urg_ptr)
    return tcp_header

print("✓ TCP header builder defined")


In [ ]:
# Step 7: Cross-Platform Transport Class

class RawTcpTransport:
    """
    Cross-platform TCP packet transport.
    - Linux/macOS: Uses raw sockets
    - Windows: Uses Scapy + Npcap fallback
    """
    
    def __init__(self, src_ip: str, dst_ip: str, src_port: int, dst_port: int, iface: Optional[str] = None):
        self.src_ip = src_ip
        self.dst_ip = dst_ip
        self.src_port = src_port
        self.dst_port = dst_port
        self.iface = iface
        self.windows_fallback = IS_WINDOWS
        
        if not self.windows_fallback:
            # Linux/macOS: Use raw socket
            self.sock = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_RAW)
            print("Using raw socket (Linux/macOS mode)")
        else:
            # Windows: Use Scapy
            if not HAVE_SCAPY:
                raise RuntimeError(
                    f"Windows detected but Scapy is not available: {SCAPY_IMPORT_ERR}.\n"
                    "Install with: pip install scapy\n"
                    "Ensure Npcap is installed with loopback support."
                )
            print("Using Scapy (Windows mode)")
    
    def encapsulate(self, data: bytes, flags: int = 0x02) -> bytes:
        """Manually encapsulate data into IP+TCP packet."""
        tcp = build_tcp_header(self.src_ip, self.dst_ip, self.src_port, self.dst_port, data, flags=flags)
        ip = build_ip_header(self.src_ip, self.dst_ip, len(tcp) + len(data))
        return ip + tcp + data
    
    def send(self, data: bytes, flags: int = 0x02):
        """Send encapsulated packet."""
        if not self.windows_fallback:
            # Linux/macOS: Send raw packet
            pkt = self.encapsulate(data, flags=flags)
            self.sock.sendto(pkt, (self.dst_ip, 0))
        else:
            # Windows: Use Scapy
            scapy_pkt = (SCAPY_IP(src=self.src_ip, dst=self.dst_ip) / 
                        SCAPY_TCP(sport=self.src_port, dport=self.dst_port, flags=flags) / 
                        SCAPY_Raw(data))
            chosen_iface = self.iface
            if chosen_iface is None and self.dst_ip in ("127.0.0.1", "::1"):
                chosen_iface = "Npcap Loopback Adapter"
            scapy_send(scapy_pkt, verbose=False, iface=chosen_iface)

print("✓ RawTcpTransport class defined")


In [ ]:
# Step 8: List Available Network Interfaces (Windows)
if IS_WINDOWS and HAVE_SCAPY:
    print("=== Available Network Interfaces ===")
    try:
        interfaces = get_if_list()
        for i, iface in enumerate(interfaces):
            print(f"  {i+1}. {iface}")
        print("\n→ For loopback traffic, use 'Npcap Loopback Adapter' or '\\Device\\NPF_Loopback'")
    except Exception as e:
        print(f"Could not list interfaces: {e}")
else:
    print("(Interface listing only available on Windows with Scapy)")


In [ ]:
# Step 9: Preview Packet Structure (Hexdump)
print("=== Packet Structure Preview ===")
print("\nBuilding a sample packet with:")
print("  Source IP: 127.0.0.1")
print("  Dest IP:   127.0.0.1")
print("  Src Port:  Random (1024-65535)")
print("  Dst Port:  12345")
print("  Payload:   'Hello from Adir & Liav!'")
print()

preview_src_ip = '127.0.0.1'
preview_dst_ip = '127.0.0.1'
preview_src_port = random.randint(1024, 65535)
preview_dst_port = 12345
preview_payload = b'Hello from Adir & Liav!'

# Build the packet
preview_tcp = build_tcp_header(preview_src_ip, preview_dst_ip, preview_src_port, preview_dst_port, preview_payload)
preview_ip = build_ip_header(preview_src_ip, preview_dst_ip, len(preview_tcp) + len(preview_payload))
preview_packet = preview_ip + preview_tcp + preview_payload

print("=== IP Header (20 bytes) ===")
hexdump(preview_ip)
print("\n=== TCP Header (20 bytes) ===")
hexdump(preview_tcp)
print("\n=== Payload ===")
hexdump(preview_payload)
print("\n=== Complete Packet ===")
hexdump(preview_packet)
print(f"\nTotal packet size: {len(preview_packet)} bytes")


## Step 10 — Create Transport Instance

⚠️ **Before running this cell:**
1. Open Wireshark
2. Start capturing on **Npcap Loopback Adapter**
3. Apply filter: `tcp.port == 12345`


In [ ]:
# Configure transport parameters
SRC_IP = '127.0.0.1'
DST_IP = '127.0.0.1'
SRC_PORT = random.randint(1024, 65535)
DST_PORT = 12345

# Interface for Windows loopback
INTERFACE = "\\Device\\NPF_Loopback"  # or "Npcap Loopback Adapter"

print(f"=== Transport Configuration ===")
print(f"  Source IP:   {SRC_IP}")
print(f"  Dest IP:     {DST_IP}")
print(f"  Source Port: {SRC_PORT}")
print(f"  Dest Port:   {DST_PORT}")
print(f"  Interface:   {INTERFACE}")

# Create transport instance
transport = RawTcpTransport(SRC_IP, DST_IP, SRC_PORT, DST_PORT, iface=INTERFACE)
print("\n✓ Transport instance created successfully!")


In [ ]:
# Step 11: Demo Send Function
def demo_send(num_packets: int = 3, delay_sec: float = 1.0, flags: int = 0x02):
    """
    Send demo packets to verify Wireshark capture is working.
    
    Args:
        num_packets: Number of packets to send
        delay_sec: Delay between packets (seconds)
        flags: TCP flags (0x02=SYN, 0x18=PSH+ACK, 0x10=ACK, 0x01=FIN)
    """
    flag_names = {
        0x02: "SYN",
        0x10: "ACK",
        0x18: "PSH+ACK",
        0x12: "SYN+ACK",
        0x01: "FIN",
        0x04: "RST"
    }
    flag_name = flag_names.get(flags, hex(flags))
    
    print(f"Sending {num_packets} demo packets with flags={flag_name}...")
    for i in range(num_packets):
        payload = f'Demo Packet {i+1} from Adir & Liav'.encode()
        transport.send(payload, flags=flags)
        print(f"  → Sent packet {i+1}/{num_packets}")
        if i < num_packets - 1:
            time.sleep(delay_sec)
    print("✓ Demo complete! Check Wireshark.")

print("✓ demo_send() function defined")


In [ ]:
# === RUN THIS TO TEST WIRESHARK CAPTURE ===
# Make sure Wireshark is capturing before running!

demo_send(num_packets=3, delay_sec=0.5, flags=0x18)  # PSH+ACK flags


## Step 12 — Send Messages from CSV File

This is the main part of the project. We iterate over each row in the CSV file and send the message as a TCP packet.


In [ ]:
def send_csv_messages(df: pd.DataFrame, delay: float = 0.1):
    """
    Send all messages from the CSV file as TCP packets.
    
    This demonstrates the TCP/IP encapsulation process:
    1. Application Layer: Message from CSV
    2. Transport Layer: TCP header added (ports, seq, ack, flags, checksum)
    3. Internet Layer: IP header added (addresses, TTL, checksum)
    4. Link Layer: Handled by OS/Npcap
    """
    print(f"=== Sending {len(df)} messages from CSV ===")
    print(f"    Source: {transport.src_ip}:{transport.src_port}")
    print(f"    Dest:   {transport.dst_ip}:{transport.dst_port}")
    print()
    
    for index, row in df.iterrows():
        msg_id = row.get('msg_id', index)
        message = row['message']
        app_protocol = row.get('app_protocol', 'HTTP')
        
        # Use message from CSV, or generate one if empty
        if not message:
            message = f"Message {msg_id}"
        
        # Encode and send
        payload = message.encode('utf-8')
        
        print(f"[{msg_id:02d}] {app_protocol}: {message[:50]}{'...' if len(message) > 50 else ''}")
        
        # Send with PSH+ACK flags (typical for data transmission)
        transport.send(payload, flags=0x18)
        
        time.sleep(delay)
    
    print()
    print(f"✓ Sent all {len(df)} messages!")
    print("  Check Wireshark to see the captured packets.")

print("✓ send_csv_messages() function defined")


In [ ]:
# === RUN THIS TO SEND ALL CSV MESSAGES ===
# Make sure Wireshark is capturing before running!

send_csv_messages(messages_df, delay=0.2)


## Step 13 — Wireshark Analysis

### Suggested Wireshark Filters

| Filter | Description |
|--------|-------------|
| `ip.addr == 127.0.0.1 && tcp.port == 12345` | All traffic on our port |
| `tcp.flags.syn == 1 && tcp.flags.ack == 0` | SYN packets only |
| `tcp.flags.push == 1 && tcp.flags.ack == 1` | PSH+ACK (data packets) |
| `tcp.payload` | Packets with payload |

### What to Look For

1. **IP Layer**: Source/Destination IPs, TTL, Protocol field (6=TCP)
2. **TCP Layer**: Ports, Flags, Sequence numbers, Checksums
3. **Payload**: Your message content (visible in ASCII)

### Save Your Capture

1. Stop the capture in Wireshark
2. Go to File → Save As
3. Save as `adir_liav_capture.pcap`


---

## Deliverables Checklist

- [x] CSV input file (`adir_liav_http_input.csv`)
- [ ] Executed notebook (with outputs) - **Save after running all cells**
- [ ] Wireshark `.pcap` capture
- [ ] Report explaining encapsulation (with screenshots)

---

## Summary: TCP/IP Encapsulation Process

```
┌─────────────────────────────────────────────────────────┐
│ Application Layer                                        │
│ ┌───────────────────────────────────────────────────┐   │
│ │ Message: "GET / HTTP/1.1"                         │   │
│ └───────────────────────────────────────────────────┘   │
│                         ↓                                │
├─────────────────────────────────────────────────────────┤
│ Transport Layer (TCP)                                    │
│ ┌─────────────────────────────────────────────────────┐ │
│ │ TCP Header (20 bytes) │ Payload                     │ │
│ │ Src Port: 51874       │ "GET / HTTP/1.1"            │ │
│ │ Dst Port: 80          │                             │ │
│ │ Flags: PSH+ACK        │                             │ │
│ └─────────────────────────────────────────────────────┘ │
│                         ↓                                │
├─────────────────────────────────────────────────────────┤
│ Internet Layer (IP)                                      │
│ ┌─────────────────────────────────────────────────────┐ │
│ │ IP Header (20 bytes) │ TCP Segment                  │ │
│ │ Src: 127.0.0.1       │ (header + payload)           │ │
│ │ Dst: 127.0.0.1       │                              │ │
│ │ Protocol: TCP (6)    │                              │ │
│ └─────────────────────────────────────────────────────┘ │
│                         ↓                                │
├─────────────────────────────────────────────────────────┤
│ Link Layer (Ethernet/Loopback)                           │
│ ┌─────────────────────────────────────────────────────┐ │
│ │ Frame │ IP Packet                         │ FCS     │ │
│ │ Header│ (header + TCP segment)            │         │ │
│ └─────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────┘
```

---

**Project by:** Adir Buksila & Liav Wizman
